# Soyuz: генерация датасета в Google Colab

Ноутбук запускает локальный открытый checkpoint DeepSeek-R1-Distill-Qwen-7B в 4-битном режиме и не использует платный DeepSeek API. Для бесплатного Colab при нехватке памяти можно заменить `MODEL_ID` на `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B`.

Перед запуском загрузи `ai-service/dataset/seed.jsonl`. Все сгенерированные строки остаются `unreviewed`: их нужно проверить человеком перед обучением. Не загружай в Colab персональные данные, секреты или реальные сообщения пользователей без отдельного разрешения.

In [ ]:
!pip -q install -U "transformers>=4.45" accelerate bitsandbytes sentencepiece


In [ ]:
import json
import os
import re
import uuid
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError("В Colab включи Runtime -> Change runtime type -> T4/L4 GPU")

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

MODEL_ID = os.environ.get(
    "MODEL_ID",
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
)
COUNTS = {
    "translate": 10,
    "assist": 10,
    "summary": 5,
    "document_analysis": 5,
}
DRIVE_OUTPUT = Path("/content/drive/MyDrive/soyuz_ai_dataset/generated")
print("Model:", MODEL_ID)
print("Counts:", COUNTS)


In [ ]:
from google.colab import drive, files

drive.mount("/content/drive")
uploaded = files.upload()
jsonl_names = [name for name in uploaded if name.endswith(".jsonl")]
if not jsonl_names:
    raise ValueError("Загрузи файл seed.jsonl из репозитория Soyuz")
SEED_PATH = Path("/content") / jsonl_names[0]
print("Seed:", SEED_PATH)


In [ ]:
SUPPORTED_LANGUAGES = ["ru", "en", "de", "fr", "es", "zh", "ar"]
ASSIST_ACTIONS = ["shorten", "formal", "friendly"]

with SEED_PATH.open(encoding="utf-8") as handle:
    seed_records = [json.loads(line) for line in handle if line.strip()]

def build_prompt(task: str, count: int) -> str:
    examples = [row for row in seed_records if row.get("task") == task][:3]
    examples_text = json.dumps(examples, ensure_ascii=False)
    return f"""Создай {count} новых качественных примеров для задачи {task}.
Поддерживаемые языки: {', '.join(SUPPORTED_LANGUAGES)}.
Для assist разрешены только действия: {', '.join(ASSIST_ACTIONS)}.
Верни только один JSON-объект вида {{\"examples\": [{{\"input\": {{...}}, \"output\": {{...}}}}]}}.
Не добавляй markdown, рассуждения, персональные данные, токены или секреты.
Не копируй дословно примеры и не выдумывай факты.
Вот несколько примеров формата: {examples_text}
"""


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model.eval()
print("Model loaded")


In [ ]:
def extract_json(text: str) -> dict:
    start = text.find("{")
    if start < 0:
        raise ValueError("Ответ модели не содержит JSON-объект")
    parsed, _ = json.JSONDecoder().raw_decode(text[start:])
    if not isinstance(parsed, dict) or not isinstance(parsed.get("examples"), list):
        raise ValueError("JSON должен содержать массив examples")
    return parsed

def generate_batch(task: str, count: int) -> dict:
    prompt = build_prompt(task, count)
    rendered = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(rendered, return_tensors="pt")
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=True,
            temperature=0.6,
            top_p=0.95,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = generated[0][inputs["input_ids"].shape[1]:]
    return extract_json(tokenizer.decode(new_tokens, skip_special_tokens=True))

def normalize(task: str, raw: dict, batch_id: str) -> list[dict]:
    rows = []
    for index, example in enumerate(raw["examples"], 1):
        if not isinstance(example, dict):
            continue
        if not isinstance(example.get("input"), dict) or not isinstance(example.get("output"), dict):
            continue
        rows.append({
            "id": f"{task}-{batch_id}-{index:04d}",
            "task": task,
            "split": "train",
            "input": example["input"],
            "output": example["output"],
            "metadata": {
                "source": "deepseek_local",
                "synthetic": True,
                "review_status": "unreviewed",
                "model": MODEL_ID,
                "license_reference": "https://github.com/deepseek-ai/DeepSeek-R1/blob/main/LICENSE",
            },
        })
    return rows


In [ ]:
def validate(rows: list[dict]) -> list[str]:
    errors = []
    ids = set()
    for index, row in enumerate(rows, 1):
        row_id = row.get("id", f"row-{index}")
        if row_id in ids:
            errors.append(f"duplicate id: {row_id}")
        ids.add(row_id)
        if row.get("task") not in {"translate", "assist", "summary", "document_analysis"}:
            errors.append(f"{row_id}: unsupported task")
        if row.get("split") not in {"train", "validation", "test"}:
            errors.append(f"{row_id}: unsupported split")
        if not isinstance(row.get("input"), dict) or not isinstance(row.get("output"), dict):
            errors.append(f"{row_id}: input/output must be objects")
        metadata = row.get("metadata", {})
        if metadata.get("source") != "deepseek_local":
            errors.append(f"{row_id}: source must be deepseek_local")
        if metadata.get("review_status") != "unreviewed":
            errors.append(f"{row_id}: generated rows must remain unreviewed")
        if row.get("task") == "translate":
            for field in ("text", "source_lang", "target_lang"):
                if not str(row["input"].get(field, "")).strip():
                    errors.append(f"{row_id}: missing input.{field}")
            if not str(row["output"].get("translated_text", "")).strip():
                errors.append(f"{row_id}: missing output.translated_text")
        if row.get("task") == "assist":
            if row["input"].get("action") not in set(ASSIST_ACTIONS):
                errors.append(f"{row_id}: invalid assist action")
            if not str(row["output"].get("result", "")).strip():
                errors.append(f"{row_id}: missing output.result")
    return errors


In [ ]:
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
all_rows = []

for task, count in COUNTS.items():
    batch_id = uuid.uuid4().hex[:12]
    raw = generate_batch(task, count)
    rows = normalize(task, raw, batch_id)
    errors = validate(rows)
    if errors:
        raise ValueError("\n".join(errors))
    output_path = DRIVE_OUTPUT / f"{task}-{batch_id}.jsonl"
    output_path.write_text(
        "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in rows),
        encoding="utf-8",
    )
    all_rows.extend(rows)
    print(task, len(rows), output_path)

combined_path = DRIVE_OUTPUT / "deepseek-local-generated.jsonl"
combined_path.write_text(
    "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in all_rows),
    encoding="utf-8",
)
print("Total generated:", len(all_rows))
print("Combined file:", combined_path)


## Перед обучением

1. Откройте случайную выборку каждой задачи и исправьте ошибки вручную.
2. Перенесите только проверенные строки в `validation` и `test` отдельно от `train`.
3. Сохраните модель, batch id и параметры генерации рядом с датасетом.
4. STT в этот ноутбук не включён: для него нужны реальные лицензированные аудиофайлы и транскрипции.